# 04 · Variables de fuente del off-block (16)

Indican de dónde sale el off-block registrado: si es la hora programada copiada o si el vuelo queda fuera del
emparejamiento con NM. Son el historial de emparejamiento NM del número de vuelo (`share_other`, `n_other`), la
llegada anterior al mismo stand (`prev_*`), las tasas de copia de llegadas cercanas (`arr_copy_*`), el formato del
número de vuelo (`fl_*`), la tasa histórica de copia fuera de mes (`te_*`) y el cruce de día local.

- `fuentes_{2025,2026}.parquet`: las del modelo final (`te_*` con los 12 meses de 2025, sin el mes de cada fila).
- `fuentes_{2025,2026}_hist.parquet`: las de la partición histórica (`te_*` solo con los meses de entrenamiento),
  para el cuaderno 05.

Por defecto reutilizamos las de v30.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "scripts" / "cuaderno.py").exists())
sys.path[:0] = [str(ROOT / "scripts"), str(ROOT)]
from cuaderno import elegir_periodo, existen, interruptor, run  # noqa: E402

PERIODO = elegir_periodo("ranking")   # abierto suelto: "ranking" (ene + jul 2026) o "final" (ene, feb, jun y jul 2026)
FINAL = PERIODO == "final"
import periodo as PER  # noqa: E402  rutas del periodo: PER.RANKING (fichero a predecir), PER.FEAT (variables)

REHACER = interruptor("REHACER_FUENTES", False)

In [ ]:
if FINAL:   # share_other/n_other cambian con los meses nuevos; las _hist solo las usa el 05 (periodo ranking)
    run("scripts/preparar_final.py", "fuentes", *(["--rehacer"] if REHACER else []))
else:
    if REHACER or not existen("Data/features/fuentes_2025_hist.parquet", "Data/features/fuentes_2026_hist.parquet"):
        run("scripts/variables_fuente.py", "2,3,4,5,6,8,9,10", "hist")
    if REHACER or not existen("Data/features/fuentes_2025.parquet", "Data/features/fuentes_2026.parquet"):
        run("scripts/variables_fuente.py")
print("listo")

In [ ]:
import pandas as pd

f = pd.read_parquet(PER.FEAT / "fuentes_2026.parquet")
print(f.shape)
f.describe().T[["count", "mean", "min", "max"]].round(3)